# Pertemuan 12 - Asosiasi Data dan Sistem Rekomendasi Dasar

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Kelas:** IF401  
**Program Studi:** PJJ Informatika

## Materi
Apriori, Market Basket Analysis, dan Content-Based Filtering.


## 1. Import Library dan Generate Dataset Transaksi


In [ ]:
try:
    import mlxtend
except ImportError:
    !pip -q install mlxtend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style="whitegrid")
np.random.seed(42)

produk = ["Roti", "Selai", "Susu", "Sereal", "Telur", "Keju", "Kopi", "Gula", "Teh", "Mentega"]
transaksi = []

for i in range(50):
    item = list(np.random.choice(produk, np.random.randint(2, 6), replace=False))
    transaksi.append(item)

for i in range(20):
    if "Roti" not in transaksi[i]:
        transaksi[i].append("Roti")
    if "Selai" not in transaksi[i]:
        transaksi[i].append("Selai")

for i in range(20, 35):
    if "Kopi" not in transaksi[i]:
        transaksi[i].append("Kopi")
    if "Gula" not in transaksi[i]:
        transaksi[i].append("Gula")

df_transaksi = pd.DataFrame({"transaksi_id": [f"T{i+1:03d}" for i in range(50)], "daftar_item": transaksi})
display(df_transaksi.head(10))


## 2. Apriori dan Association Rules


In [ ]:
counter_produk = Counter()
for items in transaksi:
    counter_produk.update(items)

freq_produk = pd.DataFrame(counter_produk.items(), columns=["produk", "frekuensi"]).sort_values("frekuensi", ascending=False)
display(freq_produk)

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

freq_items = apriori(df_onehot, min_support=0.10, use_colnames=True)
freq_items = freq_items.sort_values("support", ascending=False)

rules = association_rules(freq_items, metric="confidence", min_threshold=0.50)
rules = rules[rules["lift"] > 1].copy()
rules = rules.sort_values(["lift", "confidence"], ascending=False)

rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ", ".join(list(x)))
rules["consequents_str"] = rules["consequents"].apply(lambda x: ", ".join(list(x)))

display(freq_items.head(10))
display(rules[["antecedents_str", "consequents_str", "support", "confidence", "lift"]].head(10).round(3))


## 3. Content-Based Filtering


In [ ]:
katalog = pd.DataFrame({
    "produk": produk,
    "kategori": ["Bakery", "Bakery", "Dairy", "Bakery", "Dairy", "Dairy", "Minuman", "Bumbu", "Minuman", "Dairy"],
    "harga": [15000, 18000, 12000, 25000, 24000, 30000, 22000, 14000, 16000, 28000]
})

fitur_kategori = pd.get_dummies(katalog["kategori"], prefix="kategori")
scaler = MinMaxScaler()
fitur_harga = pd.DataFrame(scaler.fit_transform(katalog[["harga"]]), columns=["harga_scaled"])
fitur_produk = pd.concat([fitur_kategori, fitur_harga], axis=1)

sim_matrix = cosine_similarity(fitur_produk)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog["produk"] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    hasil = katalog.iloc[[i for i, _ in skor]].copy()
    hasil["similarity_score"] = [score for _, score in skor]
    return hasil

display(katalog)
display(rekomendasi_serupa("Roti", top_n=3).round(3))


## 4. Perbandingan Rekomendasi


In [ ]:
produk_target = "Roti"
rules_terkait = rules[rules["antecedents"].apply(lambda x: produk_target in x)].copy()
display(rules_terkait[["consequents_str", "support", "confidence", "lift"]].head(5).round(3))

print("Rekomendasi Content-Based:")
display(rekomendasi_serupa(produk_target, top_n=3).round(3))


## Kesimpulan

Saya mempelajari association rule mining dan rekomendasi sederhana. Temuan utama adalah Apriori menemukan pola produk yang sering dibeli bersama, sedangkan Content-Based Filtering memberi rekomendasi berdasarkan kemiripan atribut produk. Keterbatasannya, dataset transaksi masih sintetis.
